# Measuring currents and voltages

Beyond the Hamiltonian, `fluxcharge` gives the **branch current** through any
element and the **voltage** across it or between any two nodes, as operators in
the reduced coordinates:

- `result.current(edge)` — current through the element on `edge`
- `result.voltage(edge)` — voltage across that element
- `result.voltage(a, b)` — node-to-node voltage `V_a − V_b`

These come from a **circuit solve** (the elements' constitutive laws + Kirchhoff)
read off the reduced state — so they are correct even for **non-reciprocal
(gyrator) circuits**, and they satisfy Kirchhoff's laws by construction. Feed any
of them to `matrix_elements` for numeric matrix elements / expectation values.

Results are in natural units (ħ = 1, G₀ = 1).

In [ ]:
import numpy as np, sympy as sp
import matplotlib
matplotlib.use("Agg")           # in a live notebook: %matplotlib inline
import matplotlib.pyplot as plt
from fluxcharge import library, Circuit

## 1. Transmon — the Josephson supercurrent

The current through the junction is the DC Josephson relation `I = E_J·sin φ`;
the node voltage is `q/C`. Both are exact operators.

In [ ]:
tr = library.transmon(); r = tr.hamiltonian(canonical=True)
print("H          :", r.H)
print("JJ current :", r.current("e1"))          # E_J*sin(phi_v2)
print("node V(v1,v2):", r.voltage("v1", "v2"))

### Matrix elements of the supercurrent

`<i|I|j>` in the transmon eigenbasis. The diagonal vanishes (⟨n|sin φ|n⟩ = 0 by
parity); the current drives 0↔1 transitions.

In [ ]:
p = {"C": 1.0, "E_J": 12.0}
me = r.matrix_elements(r.current("e1"), p, n_levels=5, cutoffs={"phi_v2": 80})
print("|<i|I_JJ|j>| =")
print(np.round(np.abs(me), 4))

## 2. Fluxonium — inductor current and a flux-biased junction

With an external flux the junction phase is biased; the current carries it
automatically. Kirchhoff's current law closes across the shared node.

In [ ]:
fx = library.fluxonium(); rf = fx.hamiltonian(canonical=True)
for e in fx.edges:
    kind = type(fx._edges[e].element).__name__
    print(f"  {e} [{kind}]: I = {rf.current(e)}")
# KCL at the shared node v2: incidence-weighted currents sum to zero
A = fx.incidence_matrix(); eidx, vidx = fx._eidx(), fx._vidx()
kcl = sum(A[eidx[e], vidx['v2']] * rf.current(e) for e in fx.edges)
print("KCL @ v2:", sp.simplify(kcl))

## 3. The circulator — currents through a *non-reciprocal* circuit

This is the payoff: a gyrator half-edge has no constitutive I–V law, but its
current is fixed by the partner port's voltage (`I₁ = −G·V₂`). Every branch
current is well-defined and **Kirchhoff's laws still close** — currents sum to
zero at every node, voltages around every loop.

In [ ]:
cir = library.circulator()
rc = cir.hamiltonian(ground="v1", open_loops="f4", canonical=True)
print("H:", rc.H, "\n")
for e in cir.edges:
    kind = type(cir._edges[e].element).__name__
    print(f"  {e} [{kind:17s}] I = {sp.simplify(rc.current(e))}")

In [ ]:
# Kirchhoff closure across the whole non-reciprocal circuit
A, B = cir.incidence_matrix(), cir.orientation_matrix()
eidx, vidx, lidx = cir._eidx(), cir._vidx(), cir._lidx()
kcl = [sp.simplify(sum(A[eidx[e], vidx[v]] * rc.current(e) for e in cir.edges))
       for v in cir.vertices]
kvl = [sp.simplify(sum(B[lidx[l], eidx[e]] * rc.voltage(e) for e in cir.edges))
       for l in cir.loops]
print("KCL at every node :", kcl)
print("KVL around every loop:", kvl)
assert all(x == 0 for x in kcl) and all(x == 0 for x in kvl)
print("Kirchhoff satisfied — including the gyrator half-edges.")

## 4. Numeric: current expectation vs a swept parameter

`matrix_elements` returns `<i|·|j>`; the diagonal `<n|·|n>` is the expectation in
eigenstate `n`. Here we watch the RMS junction current of the transmon's low
levels grow with `E_J` (stiffer well → larger zero-point current).

In [ ]:
tr = library.transmon(); r = tr.hamiltonian(canonical=True)
EJs = np.linspace(4.0, 40.0, 12)
rms01 = []
for EJ in EJs:
    me = r.matrix_elements(r.current("e1"), {"C": 1.0, "E_J": float(EJ)},
                           n_levels=3, cutoffs={"phi_v2": 90})
    rms01.append(abs(me[0, 1]))          # 0-1 current matrix element
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(EJs, rms01, "o-")
ax.set_xlabel("E_J"); ax.set_ylabel("|<0|I_JJ|1>|"); ax.set_title("Transmon 0-1 supercurrent matrix element")
fig.tight_layout()
fig

**Takeaway.** `current` / `voltage` turn the reduced Hamiltonian into the
measurable branch quantities — the Josephson supercurrent, node voltages, and
the gyrator currents of a non-reciprocal circuit — as exact operators you can
diagonalize, take matrix elements of, or feed to QuTiP.